# 05 — TabPFN Benchmark on Athlete-Disjoint Landmark Data

This notebook evaluates TabPFN as an additional fixed model-family benchmark for the canonical CUM+DYN landmark task.

## Scope

The scientific target, cohort, landmarks, feature representation, and outer athlete-disjoint folds are inherited unchanged from Notebooks 03 and 04.

TabPFN is evaluated on:

- Team A only;
- six fixed elapsed-time landmarks: 10, 20, 30, 40, 50, and 60 minutes;
- the frozen 63-feature CUM+DYN representation;
- the same five athlete-disjoint outer folds used by the baseline models.

The benchmark is intended to test whether a modern tabular foundation model improves discrimination of the same session-level injury-associated target. It does **not** create new biological information, localize injury onset, or convert the study into minute-specific prospective risk prediction.

## Reproducibility and security

The Prior Labs access token is requested interactively with hidden input and is never written to disk or printed. Baseline predictions are loaded from Notebook 04 rather than recomputed here, ensuring that all cross-model comparisons use one canonical baseline provenance source.


In [ ]:
from pathlib import Path
import gc
import getpass
import os
import platform
import random

import numpy as np
import pandas as pd
import sklearn
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from tabpfn import TabPFNClassifier

RANDOM_STATE = 42
LANDMARKS = [10, 20, 30, 40, 50, 60]
N_BOOT = 1000

pd.set_option("display.max_columns", 200)

print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU",
)


## 1. Repository paths and canonical inputs

Notebook 03 provides the modelling table and ordered 63-feature artifact. Notebook 04 provides the frozen outer-fold assignments and current-environment baseline OOF predictions.


In [ ]:
def find_project_root(start: Path) -> Path:
    """Resolve the repository root when run from the root or notebooks directory."""
    start = start.resolve()
    if start.name.lower() == "notebooks":
        return start.parent
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

MODELLING_DIR = PROJECT_ROOT / "results" / "modelling_data"
BASELINE_DIR = PROJECT_ROOT / "results" / "baseline"
OUTPUT_DIR = PROJECT_ROOT / "results" / "tabpfn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODELLING_DIR / "model_df_2020.csv"
FEATURE_FILE = MODELLING_DIR / "primary_cum_dyn_features_2020.csv"
FOLD_FILE = BASELINE_DIR / "outer_fold_athlete_assignments.csv"
BASELINE_OOF_FILE = BASELINE_DIR / "model_benchmark_oof_predictions.csv"
BASELINE_RESULTS_FILE = BASELINE_DIR / "model_benchmark_results.csv"

for required_file in [
    MODEL_FILE,
    FEATURE_FILE,
    FOLD_FILE,
    BASELINE_OOF_FILE,
    BASELINE_RESULTS_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            "A required upstream artifact is missing. "
            "Run Notebooks 03 and 04 successfully before Notebook 05. "
            f"Missing: {required_file}"
        )

print("Project root:", PROJECT_ROOT)
print("TabPFN output directory:", OUTPUT_DIR)


## 2. Secure TabPFN authentication

The token is entered interactively and stored only in the current process environment. The notebook does not echo the token or call an endpoint that reveals account identifiers.


In [ ]:
if not os.environ.get("TABPFN_TOKEN"):
    os.environ["TABPFN_TOKEN"] = getpass.getpass(
        "Enter Prior Labs TABPFN token (hidden): "
    ).strip()

assert os.environ.get("TABPFN_TOKEN"), (
    "TABPFN_TOKEN is empty."
)

print("TabPFN token is available in the current process.")


## 3. Load and validate the canonical cohort, features, folds, and baseline predictions


In [ ]:
df = pd.read_csv(MODEL_FILE, low_memory=False)

feature_table = (
    pd.read_csv(FEATURE_FILE)
    .sort_values("feature_order")
    .reset_index(drop=True)
)

primary_features = feature_table["feature_name"].tolist()

fold_assignment = pd.read_csv(FOLD_FILE)
baseline_oof_all = pd.read_csv(BASELINE_OOF_FILE)
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)

assert len(primary_features) == 63
assert len(set(primary_features)) == 63
assert set(primary_features).issubset(df.columns)

df["team"] = (
    df["player_name"]
    .astype(str)
    .str.split("-", n=1)
    .str[0]
)

session_table = (
    df[["player_name", "session_id", "injury", "team"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

team_a_sessions = (
    session_table.loc[
        session_table["team"] == "TeamA"
    ]
    .copy()
)

assert len(session_table) == 3_743
assert session_table["player_name"].nunique() == 48
assert int(session_table["injury"].sum()) == 22
assert len(team_a_sessions) == 2_259
assert team_a_sessions["player_name"].nunique() == 27
assert int(team_a_sessions["injury"].sum()) == 22

assert len(fold_assignment) == 27
assert fold_assignment["player_name"].nunique() == 27
assert fold_assignment["test_fold"].nunique() == 5
assert (
    fold_assignment
    .groupby("test_fold")["positive_athlete"]
    .sum()
    .eq(1)
    .all()
)

print("Canonical cohort, feature, and fold artifacts validated.")


## 4. Reconstruct the five outer folds from the frozen fold artifact

No fold is regenerated from scratch in this notebook. The exact assignments exported by Notebook 04 are loaded and converted into train/test athlete lists.


In [ ]:
team_a_athletes = sorted(
    team_a_sessions["player_name"].unique()
)

outer_folds = []

for fold_id in sorted(
    fold_assignment["test_fold"].unique()
):
    test_athletes = (
        fold_assignment.loc[
            fold_assignment["test_fold"] == fold_id,
            "player_name",
        ]
        .sort_values()
        .tolist()
    )

    train_athletes = [
        athlete
        for athlete in team_a_athletes
        if athlete not in test_athletes
    ]

    assert set(train_athletes).isdisjoint(
        set(test_athletes)
    )

    outer_folds.append(
        {
            "fold": int(fold_id),
            "train_athletes": train_athletes,
            "test_athletes": test_athletes,
        }
    )

assert len(outer_folds) == 5

print("Frozen outer folds loaded successfully.")


## 5. Fixed landmark datasets

Each dataset contains at most one observation per athlete-session and retains all 22 positive Team-A athlete-sessions.


In [ ]:
team_a_athlete_set = set(team_a_athletes)

landmark_datasets = {}
landmark_rows = []

for landmark in LANDMARKS:
    dx = (
        df.loc[
            (df["minute_idx"] == landmark)
            & (df["player_name"].isin(team_a_athlete_set)),
            ["player_name", "session_id", "injury"]
            + primary_features,
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert not dx.duplicated(
        ["player_name", "session_id"]
    ).any()

    assert int(dx["injury"].sum()) == 22
    assert (
        dx.loc[
            dx["injury"] == 1,
            "player_name",
        ].nunique()
        == 5
    )

    missing_values = int(
        dx[primary_features].isna().sum().sum()
    )

    assert missing_values == 0, (
        f"TabPFN input contains missing values at "
        f"{landmark} minutes: {missing_values}"
    )

    landmark_datasets[landmark] = dx

    landmark_rows.append(
        {
            "landmark": landmark,
            "athlete_sessions": len(dx),
            "athletes": dx["player_name"].nunique(),
            "positive_sessions": int(dx["injury"].sum()),
        }
    )

landmark_summary = pd.DataFrame(landmark_rows)
display(landmark_summary)


## 6. Deterministic seed helper and device selection

GPU execution is used when CUDA is available. Seeds are reset before each fold-level fit to reduce avoidable run-to-run variation. GPU/library-level nondeterminism may still exist and is documented through saved runtime metadata.


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


print("TabPFN device:", DEVICE)


## 7. One-fold safety test

This small execution verifies athlete isolation, feature dimensionality, class presence, and TabPFN inference before the full 30-fit benchmark.


In [ ]:
landmark = 30
fold = outer_folds[0]
dx = landmark_datasets[landmark]

train = dx.loc[
    dx["player_name"].isin(fold["train_athletes"])
].copy()

test = dx.loc[
    dx["player_name"].isin(fold["test_athletes"])
].copy()

assert set(train["player_name"]).isdisjoint(
    set(test["player_name"])
)

X_train = train[primary_features]
y_train = train["injury"].astype(int).to_numpy()

X_test = test[primary_features]
y_test = test["injury"].astype(int).to_numpy()

assert X_train.shape[1] == 63
assert X_test.shape[1] == 63
assert len(np.unique(y_train)) == 2
assert len(np.unique(y_test)) == 2

set_all_seeds(RANDOM_STATE + landmark + fold["fold"])

if torch.cuda.is_available():
    torch.cuda.empty_cache()

smoke_model = TabPFNClassifier(device=DEVICE)
smoke_model.fit(X_train, y_train)
smoke_prob = smoke_model.predict_proba(X_test)[:, 1]

print("Landmark:", landmark)
print("Fold:", fold["fold"])
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train positives:", int(y_train.sum()))
print("Test positives:", int(y_test.sum()))
print("ROC-AUC:", roc_auc_score(y_test, smoke_prob))
print("PR-AUC:", average_precision_score(y_test, smoke_prob))
print("TabPFN one-fold safety test passed.")

del smoke_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 8. Full TabPFN benchmark

Thirty independent outer-fold fits are performed: six landmarks × five athlete-disjoint folds. Every evaluable athlete-session receives exactly one held-out prediction at each landmark where it is observable.


In [ ]:
tabpfn_oof = {}
tabpfn_summary_rows = []

for landmark in LANDMARKS:
    dx = landmark_datasets[landmark]
    fold_predictions = []

    print("\n" + "=" * 68)
    print(f"LANDMARK {landmark} MIN")
    print("=" * 68)

    for fold in outer_folds:
        train = dx.loc[
            dx["player_name"].isin(
                fold["train_athletes"]
            )
        ].copy()

        test = dx.loc[
            dx["player_name"].isin(
                fold["test_athletes"]
            )
        ].copy()

        assert set(train["player_name"]).isdisjoint(
            set(test["player_name"])
        )

        X_train = train[primary_features]
        y_train = train["injury"].astype(int).to_numpy()

        X_test = test[primary_features]
        y_test = test["injury"].astype(int).to_numpy()

        assert X_train.shape[1] == 63
        assert X_test.shape[1] == 63
        assert len(np.unique(y_train)) == 2
        assert len(np.unique(y_test)) == 2

        fit_seed = (
            RANDOM_STATE
            + 100 * landmark
            + fold["fold"]
        )
        set_all_seeds(fit_seed)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = TabPFNClassifier(device=DEVICE)
        model.fit(X_train, y_train)

        y_prob = model.predict_proba(
            X_test
        )[:, 1]

        fold_predictions.append(
            pd.DataFrame(
                {
                    "player_name": test["player_name"].to_numpy(),
                    "session_id": test["session_id"].to_numpy(),
                    "injury": y_test,
                    "fold": fold["fold"],
                    "landmark": landmark,
                    "y_prob": y_prob,
                }
            )
        )

        print(
            f"Fold {fold['fold']} | "
            f"train={len(train)} | "
            f"test={len(test)} | "
            f"train_pos={int(y_train.sum())} | "
            f"test_pos={int(y_test.sum())}"
        )

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    assert len(oof) == len(dx)

    assert not oof.duplicated(
        ["player_name", "session_id"]
    ).any()

    expected_keys = set(
        zip(dx["player_name"], dx["session_id"])
    )
    observed_keys = set(
        zip(oof["player_name"], oof["session_id"])
    )
    assert expected_keys == observed_keys

    roc = roc_auc_score(
        oof["injury"],
        oof["y_prob"],
    )
    pr = average_precision_score(
        oof["injury"],
        oof["y_prob"],
    )

    tabpfn_oof[landmark] = oof

    tabpfn_summary_rows.append(
        {
            "landmark": landmark,
            "athlete_sessions": len(oof),
            "positive_sessions": int(
                oof["injury"].sum()
            ),
            "roc_auc": roc,
            "pr_auc": pr,
        }
    )

    print(
        f"POOLED {landmark} min | "
        f"ROC-AUC={roc:.6f} | "
        f"PR-AUC={pr:.6f}"
    )

tabpfn_summary = pd.DataFrame(
    tabpfn_summary_rows
)

tabpfn_oof_all = pd.concat(
    [tabpfn_oof[x] for x in LANDMARKS],
    ignore_index=True,
)

display(tabpfn_summary.round(6))


## 9. Frozen-result consistency check

The values below come from the previously completed canonical TabPFN benchmark. Small numerical deviations can occur across GPU/software stacks, so this is reported as a reproducibility diagnostic rather than a hard failure criterion.


In [ ]:
FROZEN_TABPFN = pd.DataFrame(
    {
        "landmark": LANDMARKS,
        "frozen_roc_auc": [
            0.564563,
            0.635002,
            0.691560,
            0.674331,
            0.599044,
            0.610121,
        ],
        "frozen_pr_auc": [
            0.010949,
            0.013560,
            0.015424,
            0.014373,
            0.011939,
            0.012895,
        ],
    }
)

reproducibility_check = (
    tabpfn_summary[
        ["landmark", "roc_auc", "pr_auc"]
    ]
    .merge(
        FROZEN_TABPFN,
        on="landmark",
        validate="one_to_one",
    )
)

reproducibility_check["abs_delta_roc"] = (
    reproducibility_check["roc_auc"]
    - reproducibility_check["frozen_roc_auc"]
).abs()

reproducibility_check["abs_delta_pr"] = (
    reproducibility_check["pr_auc"]
    - reproducibility_check["frozen_pr_auc"]
).abs()

display(reproducibility_check.round(8))

print(
    "Maximum absolute ROC difference:",
    reproducibility_check["abs_delta_roc"].max(),
)
print(
    "Maximum absolute PR difference:",
    reproducibility_check["abs_delta_pr"].max(),
)


## 10. Validate canonical baseline OOF predictions

Baseline OOF predictions are not rebuilt here. They are loaded from Notebook 04 and checked for exact athlete-session key compatibility with the TabPFN predictions.


In [ ]:
required_baseline_models = {
    "logistic",
    "random_forest",
    "xgboost",
}

available_baseline_models = set(
    baseline_oof_all["model"].unique()
)

missing_baseline_models = (
    required_baseline_models
    - available_baseline_models
)

assert not missing_baseline_models, (
    "Canonical baseline OOF file is missing model(s): "
    f"{sorted(missing_baseline_models)}"
)

for landmark in LANDMARKS:
    tab = tabpfn_oof_all.loc[
        tabpfn_oof_all["landmark"] == landmark
    ]

    for model_name in sorted(
        required_baseline_models
    ):
        base = baseline_oof_all.loc[
            (baseline_oof_all["landmark"] == landmark)
            & (baseline_oof_all["model"] == model_name)
        ]

        assert len(tab) == len(base)

        tab_keys = set(
            zip(tab["player_name"], tab["session_id"])
        )
        base_keys = set(
            zip(base["player_name"], base["session_id"])
        )

        assert tab_keys == base_keys

print(
    "TabPFN and canonical baseline OOF keys match "
    "at all six landmarks."
)


## 11. Paired athlete-cluster bootstrap: TabPFN versus canonical baselines

Within each landmark, one shared set of athlete-bootstrap draws is used for all three model contrasts. Athlete multiplicities are represented as observation weights rather than by repeatedly concatenating duplicated DataFrames.

The reported intervals are exploratory paired uncertainty intervals. Given the very small number of positive athletes and the multiple model/landmark contrasts, interpretation should emphasize whether intervals exclude zero rather than use unqualified significance language.


In [ ]:
def paired_metrics_from_weights(
    y,
    p_tabpfn,
    p_baseline,
    sample_weight,
):
    delta_roc = (
        roc_auc_score(
            y,
            p_tabpfn,
            sample_weight=sample_weight,
        )
        - roc_auc_score(
            y,
            p_baseline,
            sample_weight=sample_weight,
        )
    )

    delta_pr = (
        average_precision_score(
            y,
            p_tabpfn,
            sample_weight=sample_weight,
        )
        - average_precision_score(
            y,
            p_baseline,
            sample_weight=sample_weight,
        )
    )

    return delta_roc, delta_pr


comparison_rows = []

for landmark in LANDMARKS:
    tab = (
        tabpfn_oof_all.loc[
            tabpfn_oof_all["landmark"] == landmark,
            ["player_name", "session_id", "injury", "y_prob"],
        ]
        .rename(columns={"y_prob": "p_tabpfn"})
    )

    athletes = np.array(
        sorted(tab["player_name"].unique())
    )

    athlete_to_index = {
        athlete: idx
        for idx, athlete in enumerate(athletes)
    }

    rng = np.random.default_rng(
        2026 + landmark
    )

    bootstrap_counts = np.vstack(
        [
            np.bincount(
                rng.integers(
                    0,
                    len(athletes),
                    size=len(athletes),
                ),
                minlength=len(athletes),
            )
            for _ in range(N_BOOT)
        ]
    )

    for model_name in [
        "logistic",
        "random_forest",
        "xgboost",
    ]:
        base = (
            baseline_oof_all.loc[
                (
                    baseline_oof_all["landmark"]
                    == landmark
                )
                & (
                    baseline_oof_all["model"]
                    == model_name
                ),
                [
                    "player_name",
                    "session_id",
                    "injury",
                    "y_prob",
                ],
            ]
            .rename(
                columns={
                    "y_prob": "p_baseline"
                }
            )
        )

        paired = tab.merge(
            base,
            on=[
                "player_name",
                "session_id",
                "injury",
            ],
            how="inner",
            validate="one_to_one",
        )

        assert len(paired) == len(tab) == len(base)

        y = paired["injury"].to_numpy()
        p_tab = paired["p_tabpfn"].to_numpy()
        p_base = paired["p_baseline"].to_numpy()

        observed_delta_roc = (
            roc_auc_score(y, p_tab)
            - roc_auc_score(y, p_base)
        )
        observed_delta_pr = (
            average_precision_score(y, p_tab)
            - average_precision_score(y, p_base)
        )

        row_athlete_index = (
            paired["player_name"]
            .map(athlete_to_index)
            .to_numpy()
        )

        boot_roc = []
        boot_pr = []

        for counts in bootstrap_counts:
            weights = counts[
                row_athlete_index
            ].astype(float)

            positive_weight = weights[y == 1].sum()
            negative_weight = weights[y == 0].sum()

            if (
                positive_weight == 0
                or negative_weight == 0
            ):
                continue

            d_roc, d_pr = paired_metrics_from_weights(
                y,
                p_tab,
                p_base,
                weights,
            )

            boot_roc.append(d_roc)
            boot_pr.append(d_pr)

        comparison_rows.append(
            {
                "baseline": model_name,
                "landmark": landmark,
                "delta_roc_tabpfn_minus_baseline": (
                    observed_delta_roc
                ),
                "roc_ci_low": np.quantile(
                    boot_roc, 0.025
                ),
                "roc_ci_high": np.quantile(
                    boot_roc, 0.975
                ),
                "delta_pr_tabpfn_minus_baseline": (
                    observed_delta_pr
                ),
                "pr_ci_low": np.quantile(
                    boot_pr, 0.025
                ),
                "pr_ci_high": np.quantile(
                    boot_pr, 0.975
                ),
                "n_valid_bootstrap": len(boot_roc),
            }
        )

paired_tabpfn_comparison = pd.DataFrame(
    comparison_rows
)

display(paired_tabpfn_comparison.round(6))


## 12. Compact interpretation table

This table is descriptive and deliberately avoids winner-take-all model selection. It reports whether each 95% paired bootstrap interval excludes zero.


In [ ]:
interpretation_table = (
    paired_tabpfn_comparison.copy()
)

interpretation_table["roc_interval_excludes_zero"] = (
    (
        interpretation_table["roc_ci_low"] > 0
    )
    | (
        interpretation_table["roc_ci_high"] < 0
    )
)

interpretation_table["pr_interval_excludes_zero"] = (
    (
        interpretation_table["pr_ci_low"] > 0
    )
    | (
        interpretation_table["pr_ci_high"] < 0
    )
)

display(
    interpretation_table[
        [
            "baseline",
            "landmark",
            "delta_roc_tabpfn_minus_baseline",
            "roc_ci_low",
            "roc_ci_high",
            "roc_interval_excludes_zero",
            "delta_pr_tabpfn_minus_baseline",
            "pr_ci_low",
            "pr_ci_high",
            "pr_interval_excludes_zero",
        ]
    ].round(6)
)


## 13. Export canonical TabPFN artifacts


In [ ]:
TABPFN_SUMMARY_FILE = (
    OUTPUT_DIR / "tabpfn_primary_summary.csv"
)
TABPFN_OOF_FILE = (
    OUTPUT_DIR / "tabpfn_primary_oof_predictions.csv"
)
PAIRED_FILE = (
    OUTPUT_DIR / "tabpfn_paired_baseline_comparison.csv"
)
REPRO_FILE = (
    OUTPUT_DIR / "tabpfn_reproducibility_check.csv"
)
RUNTIME_FILE = (
    OUTPUT_DIR / "tabpfn_runtime_versions.csv"
)

tabpfn_summary.to_csv(
    TABPFN_SUMMARY_FILE,
    index=False,
)

tabpfn_oof_all.to_csv(
    TABPFN_OOF_FILE,
    index=False,
)

paired_tabpfn_comparison.to_csv(
    PAIRED_FILE,
    index=False,
)

reproducibility_check.to_csv(
    REPRO_FILE,
    index=False,
)

runtime_versions = pd.DataFrame(
    [
        {
            "package": "python",
            "version": platform.python_version(),
        },
        {
            "package": "numpy",
            "version": np.__version__,
        },
        {
            "package": "pandas",
            "version": pd.__version__,
        },
        {
            "package": "scikit-learn",
            "version": sklearn.__version__,
        },
        {
            "package": "torch",
            "version": torch.__version__,
        },
        {
            "package": "tabpfn",
            "version": (
                __import__("tabpfn").__version__
                if hasattr(
                    __import__("tabpfn"),
                    "__version__",
                )
                else "unknown"
            ),
        },
        {
            "package": "cuda_available",
            "version": str(
                torch.cuda.is_available()
            ),
        },
        {
            "package": "gpu",
            "version": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else "CPU"
            ),
        },
    ]
)

runtime_versions.to_csv(
    RUNTIME_FILE,
    index=False,
)

print("Saved:", TABPFN_SUMMARY_FILE)
print("Saved:", TABPFN_OOF_FILE)
print("Saved:", PAIRED_FILE)
print("Saved:", REPRO_FILE)
print("Saved:", RUNTIME_FILE)


## Output contract and claims discipline

A successful run produces:

- `results/tabpfn/tabpfn_primary_summary.csv`
- `results/tabpfn/tabpfn_primary_oof_predictions.csv`
- `results/tabpfn/tabpfn_paired_baseline_comparison.csv`
- `results/tabpfn/tabpfn_reproducibility_check.csv`
- `results/tabpfn/tabpfn_runtime_versions.csv`

The appropriate interpretation is model-comparison evidence under the same severely data-limited athlete-disjoint landmark task. TabPFN may improve discrimination relative to some baselines at some landmarks, but the study does not support a claim of universal model dominance. The inferential ceiling remains the five athletes contributing positive sessions.
